# YOLOv11 Training for Car Parts Detection - Batch 2

Training YOLOv11 model on car parts dataset (Version 2) from Roboflow using Ultralytics.

## Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.8 MB/s eta 0:00:00


## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
from pathlib import Path

print("📥 Downloading car parts dataset (Version 2) from Roboflow...\n")

# Initialize Roboflow
rf = Roboflow(api_key="HTEXw6EPVueKWxC0Hh7S")
project = rf.workspace("bun-7he2f").project("deteccion-partes-789av")
version = project.version(2)

# Download in YOLOv11 format
dataset = version.download("yolov11")

# Dataset location
DATASET_ROOT = Path(dataset.location)
print(f"\n✅ Dataset downloaded to: {DATASET_ROOT}")
print(f"\n📂 Dataset structure:")
!ls -lh {DATASET_ROOT}

# Check data.yaml
DATA_YAML = DATASET_ROOT / "data.yaml"
print(f"\n📄 Dataset config (data.yaml):")
!cat {DATA_YAML}

📥 Downloading car parts dataset (Version 2) from Roboflow...

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to deteccion-partes-2 in yolov11:: 100%|██████████| 13425/13425 [00:02<00:00, 4846.07it/s]



✅ Dataset downloaded to: /content/deteccion-partes-2

📂 Dataset structure:
total 24K
-rw-r--r-- 1 root root  545 May  7 13:41 data.yaml
-rw-r--r-- 1 root root  148 May  7 13:41 README.dataset.txt
-rw-r--r-- 1 root root 1.3K May  7 13:41 README.roboflow.txt
drwxr-xr-x 4 root root 4.0K May  7 13:41 test
drwxr-xr-x 4 root root 4.0K May  7 13:41 train
drwxr-xr-x 4 root root 4.0K May  7 13:41 valid

📄 Dataset config (data.yaml):
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 21
names: ['Back-bumper', 'Back-door', 'Back-wheel', 'Back-window', 'Back-windshield', 'Fender', 'Front-bumper', 'Front-door', 'Front-wheel', 'Front-window', 'Grille', 'Headlight', 'Hood', 'License-plate', 'Mirror', 'Quarter-panel', 'Rocker-panel', 'Roof', 'Tail-light', 'Trunk', 'Windshield']

roboflow:
  workspace: bun-7he2f
  project: deteccion-partes-789av
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/bun-7he2f/deteccion-partes-789av/dataset/2

## Configure Training Parameters

In [ ]:
from pathlib import Path
import yaml

# Load dataset config
DATA_YAML = Path(dataset.location) / "data.yaml"

with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

NUM_CLASSES = data_config['nc']
CLASS_NAMES = data_config['names']

# Training parameters
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16
MODEL_SIZE = 'yolo11n'  # nano model (fastest, good for detection)

# Working directory
WORK_DIR = Path("/content/work_dirs/yolov11_batch2")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Dataset Info:")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Class names: {CLASS_NAMES[:5]}... (showing first 5)")
print(f"\n🏋️ Training Config:")
print(f"  Model: {MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Working directory: {WORK_DIR}")

📊 Dataset Info:
  Classes: 21
  Class names: ['Back-bumper', 'Back-door', 'Back-wheel', 'Back-window', 'Back-windshield']... (showing first 5)

🏋️ Training Config:
  Model: yolo11n
  Epochs: 100
  Batch size: 16
  Image size: 640x640
  Working directory: /content/work_dirs/yolov11_batch2


## Train YOLOv11 Model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11 model
model = YOLO(f'{MODEL_SIZE}.pt')

print(f"\n🚀 Starting training...\n")

# Train the model
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name='yolov11_car_parts_batch2',
    project=str(WORK_DIR),
    patience=50,  # Early stopping patience
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    device=0,  # GPU 0
    workers=8,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    deterministic=False,
    single_cls=False,
    rect=False,
    cos_lr=True,
    close_mosaic=10,
    resume=False,
    amp=True,  # Automatic Mixed Precision
    fraction=1.0,
    profile=False,
    freeze=None,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    pose=12.0,
    kobj=1.0,
    label_smoothing=0.0,
    nbs=64,
    overlap_mask=True,
    mask_ratio=4,
    dropout=0.0,
    val=True
)

print("\n✅ Training completed!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Starting training...

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/deteccion-partes-2/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5,

## Validate Best Model

In [ ]:
# Load best model
best_model_path = WORK_DIR / "yolov11_car_parts_batch2" / "weights" / "best.pt"

print(f"📊 Validating best model: {best_model_path}\n")

model = YOLO(str(best_model_path))
metrics = model.val()

print(f"\n✅ Validation Results:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

📊 Validating best model: /content/work_dirs/yolov11_batch2/yolov11_car_parts_batch2/weights/best.pt

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,586,247 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1807.2±858.6 MB/s, size: 62.1 KB)
val: Scanning /content/deteccion-partes-2/valid/labels.cache... 419 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 419/419 159.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.7it/s 10.1s
                   all        419       6705      0.927      0.883      0.933       0.77
           Back-bumper        188        190      0.941      0.821      0.905      0.768
             Back-door        309        315      0.981      0.959      0.974      0.871
            Back-wheel        359        363      0.968      0.994      0.993      0.847
           Back-windo

## Export Model

In [ ]:
# Export to ONNX format (optional)
print("📦 Exporting model to ONNX format...\n")

model.export(format='onnx', imgsz=IMG_SIZE)

print("\n✅ Model exported successfully!")

📦 Exporting model to ONNX format...

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/work_dirs/yolov11_batch2/yolov11_car_parts_batch2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 25, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 301ms
Prepared 4 packages in 7.02s
Installed 4 packages in 311ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 8.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with on

## Save Results to Google Drive

In [ ]:
import shutil
from pathlib import Path
import zipfile
from tqdm import tqdm

TRAIN_DIR = WORK_DIR / "yolov11_car_parts_batch2"

if TRAIN_DIR.exists():
    # Calculate total size
    print("📊 Calculating folder size...")
    total_size = sum(f.stat().st_size for f in TRAIN_DIR.rglob("*") if f.is_file())
    total_mb = total_size / (1024*1024)
    file_count = sum(1 for f in TRAIN_DIR.rglob("*") if f.is_file())

    print(f"   Total: {file_count} files, {total_mb:.2f} MB")

    # Create zip archive
    print("\n📦 Creating zip archive...")
    zip_path = '/content/yolov11_batch2_training.zip'

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        files_list = list(TRAIN_DIR.rglob("*"))
        for file in tqdm(files_list, desc="Zipping files", unit="file"):
            if file.is_file():
                arcname = file.relative_to(TRAIN_DIR.parent)
                zipf.write(file, arcname)

    zip_size = Path(zip_path).stat().st_size / (1024*1024)
    print(f"✅ Zip created: {zip_size:.2f} MB")

    # Copy to Google Drive
    print("\n📤 Copying to Google Drive...")
    DRIVE_PATH = Path("/content/drive/MyDrive")

    shutil.copy2(zip_path, DRIVE_PATH / 'yolov11_batch2_training.zip')

    print(f"✅ Saved to Drive: {DRIVE_PATH}/yolov11_batch2_training.zip")

    # Download to PC
    print("\n💾 Downloading to your PC...")
    from google.colab import files
    files.download(zip_path)
    print("✅ Download started! Check your browser downloads.")

    # Summary
    print(f"\n📊 Summary:")
    print(f"   Files: {file_count}")
    print(f"   Original: {total_mb:.2f} MB")
    print(f"   Compressed: {zip_size:.2f} MB")
    print(f"   Saved {(1 - zip_size/total_mb)*100:.1f}% space")
else:
    print("❌ Training folder doesn't exist yet!")

📊 Calculating folder size...
   Total: 35 files, 234.71 MB

📦 Creating zip archive...


Zipping files: 100%|██████████| 36/36 [00:13<00:00,  2.69file/s]


✅ Zip created: 211.10 MB

📤 Copying to Google Drive...
✅ Saved to Drive: /content/drive/MyDrive/yolov11_batch2_training.zip

💾 Downloading to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started! Check your browser downloads.

📊 Summary:
   Files: 35
   Original: 234.71 MB
   Compressed: 211.10 MB
   Saved 10.1% space


## Training Complete!

Your YOLOv11 model has been trained on the car parts dataset (Batch 2).

### Results saved to:
- Google Drive: `/content/drive/MyDrive/yolov11_batch2_training.zip`
- Local download: Check your browser downloads

### Files included:
- `weights/best.pt` - Best model checkpoint
- `weights/last.pt` - Last epoch checkpoint
- `results.csv` - Training metrics
- `results.png` - Training curves
- `confusion_matrix.png` - Confusion matrix
- `val_batch*.jpg` - Validation predictions